In [0]:
%sql
SELECT 
  d.fiscal_year,
  d.quarter_name,
  COUNT(DISTINCT c.customer_id) as new_customers
FROM 03_gold.final_report.dim_customer c
JOIN 03_gold.final_report.dim_date d 
  ON c.account_created_date = d.date
WHERE c.is_current = TRUE
GROUP BY d.fiscal_year, d.quarter_name, d.quarter
ORDER BY d.fiscal_year, d.quarter

In [0]:
%sql
SELECT 
  c.country,
  c.industry_type,
  COUNT(DISTINCT c.customer_id) as churned_customers,
  ROUND(COUNT(DISTINCT c.customer_id) * 100.0 / 
    (SELECT COUNT(DISTINCT customer_id) FROM 03_gold.final_report.dim_customer), 2) as churn_rate_pct
FROM 03_gold.final_report.dim_customer c
WHERE c.is_active = FALSE 
  AND c.is_current = TRUE
GROUP BY c.country, c.industry_type
ORDER BY churned_customers DESC

In [0]:
%sql
SELECT 
  customer_name,
  country,
  industry_type,
  SUM(annual_recurring_revenue) as total_arr,
  COUNT(opportunity_id) as num_contracts
FROM 03_gold.final_report.curated_opportunity
WHERE close_status = 'Won'
GROUP BY customer_name, country, industry_type
ORDER BY total_arr DESC
LIMIT 10

In [0]:
%sql
WITH customer_product_count AS (
  SELECT 
    customer_id,
    customer_name,
    COUNT(DISTINCT product_id) as num_products,
    SUM(annual_recurring_revenue) as total_arr
  FROM 03_gold.final_report.curated_opportunity
  WHERE close_status = 'Won'
  GROUP BY customer_id, customer_name
)
SELECT 
  customer_name,
  num_products as products_purchased,
  total_arr,
  CASE 
    WHEN num_products >= 3 THEN 'High Expansion'
    WHEN num_products = 2 THEN 'Medium Expansion'
    ELSE 'Single Product'
  END as expansion_tier
FROM customer_product_count
WHERE num_products >= 2
ORDER BY num_products DESC, total_arr DESC

In [0]:
%sql
SELECT 
  customer_name,
  country,
  COUNT(opportunity_id) as active_contracts,
  SUM(monthly_recurring_revenue) as current_mrr,
  MAX(end_date) as last_contract_date,
  DATEDIFF(DATE '2024-06-01', MAX(end_date)) as days_since_last_contract
FROM 03_gold.final_report.curated_opportunity
WHERE close_status = 'Won'
GROUP BY customer_name, country
HAVING MAX(end_date) < DATE '2023-12-01'
ORDER BY days_since_last_contract DESC

In [0]:
%sql
SELECT 
  start_fiscal_year as fiscal_year,
  SUM(monthly_recurring_revenue) as total_mrr,
  SUM(annual_recurring_revenue) as total_arr,
  COUNT(DISTINCT customer_id) as active_customers
FROM 03_gold.final_report.curated_opportunity
WHERE close_status = 'Won'
  AND start_fiscal_year = 2024
GROUP BY start_fiscal_year
ORDER BY start_fiscal_year

In [0]:
%sql
SELECT 
  plan_name,
  billing_cycle,
  COUNT(opportunity_id) as num_contracts,
  SUM(revenue_amount) as total_revenue,
  SUM(monthly_recurring_revenue) as total_mrr,
  AVG(contract_duration_days) as avg_contract_days
FROM 03_gold.final_report.curated_opportunity
WHERE close_status = 'Won'
GROUP BY plan_name, billing_cycle
ORDER BY total_revenue DESC

In [0]:
%sql
WITH yearly_revenue AS (
  SELECT 
    start_year as year,
    SUM(revenue_amount) as total_revenue,
    SUM(annual_recurring_revenue) as total_arr
  FROM 03_gold.final_report.curated_opportunity
  WHERE close_status = 'Won'
  GROUP BY start_year
)
SELECT 
  year,
  total_revenue,
  total_arr,
  LAG(total_revenue) OVER (ORDER BY year) as prev_year_revenue,
  ROUND((total_revenue - LAG(total_revenue) OVER (ORDER BY year)) * 100.0 / 
    LAG(total_revenue) OVER (ORDER BY year), 2) as yoy_growth_pct
FROM yearly_revenue
ORDER BY year

In [0]:
%sql
SELECT 
  employee_name,
  employee_role,
  employee_region,
  COUNT(opportunity_id) as deals_closed,
  SUM(revenue_amount) as total_revenue,
  SUM(annual_recurring_revenue) as total_arr,
  ROUND(AVG(revenue_amount), 2) as avg_deal_size
FROM 03_gold.final_report.curated_opportunity
WHERE close_status = 'Won'
GROUP BY employee_name, employee_role, employee_region
ORDER BY total_revenue DESC
LIMIT 20

In [0]:
%sql
SELECT 
  DATE_FORMAT(start_date, 'MMM yyyy') as month_year,
  start_year as year,
  start_month as month,
  SUM(monthly_recurring_revenue) as total_mrr,
  COUNT(DISTINCT customer_id) as unique_customers,
  ROUND(SUM(monthly_recurring_revenue) / COUNT(DISTINCT customer_id), 2) as avg_mrr_per_customer
FROM 03_gold.final_report.curated_opportunity
WHERE close_status = 'Won'
  AND start_date >= DATE '2023-06-01'
  AND start_date <= DATE '2024-06-01'
GROUP BY DATE_FORMAT(start_date, 'MMM yyyy'), start_year, start_month
ORDER BY start_year, start_month